# Preparación de datos demográficos, de salud y de PIB de OWID (2000–2022)

**Objetivo:** descargar seis conjuntos de datos, revisar sus columnas, preparar cada uno por separado y unirlos por `entity` y `year`. El resultado final tiene 8 columnas: `entity`, `year`, `population`, `births`, `deaths`, `child_deaths_under_5`, `percentage_health` y `gdp`. Servirá para una práctica posterior de aprendizaje automático.

Ejecuta las celdas en orden. Requiere `pandas` y `matplotlib`, además de acceso a internet. Fuente: [Our World in Data](https://ourworldindata.org/).


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

OPCIONES_OWID = {"User-Agent": "Our World In Data data fetch/1.0"}
BASE = "https://ourworldindata.org/grapher/"
SUFIJO = ".csv?v=1&csvType=full&useColumnShortNames=true"

## 1. Descargar los seis archivos

Las direcciones son cadenas de texto simples. Cada archivo se carga en su propio DataFrame. Los dos últimos son nuevos:

- `health`: gasto total en salud como porcentaje del PIB (2000–2023).
- `gdp`: PIB per cápita ajustado por inflación y por diferencias en el costo de vida.

In [ ]:
url_population = BASE + "population-with-un-projections" + SUFIJO
url_births = BASE + "number-of-births-per-year" + SUFIJO
url_deaths = BASE + "number-of-deaths-per-year" + SUFIJO
url_child_deaths = BASE + "number-of-child-deaths-unwpp" + SUFIJO
url_health = BASE + "total-healthcare-expenditure-gdp" + SUFIJO
url_gdp = BASE + "gdp-per-capita-worldbank" + SUFIJO

population = pd.read_csv(url_population, storage_options=OPCIONES_OWID)
births = pd.read_csv(url_births, storage_options=OPCIONES_OWID)
deaths = pd.read_csv(url_deaths, storage_options=OPCIONES_OWID)
child_deaths = pd.read_csv(url_child_deaths, storage_options=OPCIONES_OWID)
health = pd.read_csv(url_health, storage_options=OPCIONES_OWID)
gdp = pd.read_csv(url_gdp, storage_options=OPCIONES_OWID)

print("Filas y columnas:")
print("population:", population.shape)
print("births:", births.shape)
print("deaths:", deaths.shape)
print("child_deaths:", child_deaths.shape)
print("health:", health.shape)
print("gdp:", gdp.shape)

## 2. Explorar las columnas

OWID puede modificar los nombres de los indicadores. Revisamos las columnas reales antes de seleccionarlas.

In [ ]:
for nombre, tabla in [("population", population),
                      ("births", births),
                      ("deaths", deaths),
                      ("child_deaths", child_deaths),
                      ("health", health),
                      ("gdp", gdp)]:
    print(f"\n{nombre}: {tabla.columns.tolist()}")
    display(tabla.head(3))

## 3. Preparar población

Elegimos la columna de **estimaciones históricas** para ambos sexos y todas las edades; excluimos las proyecciones. Después conservamos los años de interés.

In [ ]:
columnas_population = [
    columna for columna in population.columns
        if "population" in columna and "estimates" in columna]

print("Columnas candidatas:", columnas_population)

assert len(columnas_population) == 1, "Revisar manualmente el indicador de población mostrado arriba"


col_population = columnas_population[0]

population_limpio = population[["entity", "year", col_population]].copy()

population_limpio = population_limpio.rename(columns={col_population: "population"})

population_limpio = population_limpio[population_limpio["year"].between(2000, 2022)].copy()

print(population_limpio.shape)
display(population_limpio.head())

## 4. Preparar nacimientos

Seleccionamos el número anual de nacimientos históricos. Revisamos qué entidades comparte con las otras tablas.

In [ ]:
columnas_births = [c for c in births.columns
                  if "births" in c and "estimates" in c]


print("Columnas candidatas:", columnas_births)
assert len(columnas_births) == 1, "Revisar manualmente el indicador de nacimientos mostrado arriba"
col_births = columnas_births[0]

births_limpio = births[["entity", "year", col_births]].copy()
births_limpio = births_limpio.rename(columns={col_births: "births"})
births_limpio = births_limpio[births_limpio["year"].between(2000, 2022)].copy()
print(births_limpio.shape)
display(births_limpio.head())

## 5. Preparar defunciones totales

In [ ]:
columnas_deaths = [c for c in deaths.columns
                   if "deaths" in c and "estimates" in c]

print("Columnas candidatas:", columnas_deaths)
assert len(columnas_deaths) == 1, "Revisar manualmente el indicador de defunciones mostrado arriba"

col_deaths = columnas_deaths[0]

deaths_limpio = deaths[["entity", "year", col_deaths]].copy()
deaths_limpio = deaths_limpio.rename(columns={col_deaths: "deaths"})
deaths_limpio = deaths_limpio[deaths_limpio["year"].between(2000, 2022)].copy()

print(deaths_limpio.shape)
display(deaths_limpio.head())

## 6. Preparar defunciones de menores de cinco años

Este indicador corresponde a niños de 0 a 4 años; no debe confundirse con mortalidad de menores de un año.

In [ ]:
columnas_child = [c for c in child_deaths.columns
                  if c not in ["entity", "code", "year"]]

print("Columnas candidatas:", columnas_child)

assert len(columnas_child) == 1, "Revisar manualmente el indicador de menores de cinco años mostrado arriba"

col_child = columnas_child[0]

child_deaths_limpio = child_deaths[["entity", "year", col_child]].copy()
child_deaths_limpio = child_deaths_limpio.rename(columns={col_child: "child_deaths_under_5"})

child_deaths_limpio = child_deaths_limpio[child_deaths_limpio["year"].between(2000, 2022)].copy()

print(child_deaths_limpio.shape)
display(child_deaths_limpio.head())

## 7. Preparar gasto en salud (`percentage_health`)

Porcentaje del PIB nacional que se destina al gasto total en salud. Igual que en las tablas anteriores, tomamos la única columna de valores y conservamos los años de interés (el archivo llega hasta 2023, pero el resto de las tablas termina en 2022).

In [ ]:
columnas_health = [c for c in health.columns
                   if c not in ["entity", "code", "year"]]

print("Columnas candidatas:", columnas_health)
assert len(columnas_health) == 1, "Revisar manualmente el indicador de gasto en salud mostrado arriba"

col_health = columnas_health[0]

health_limpio = health[["entity", "year", col_health]].copy()
health_limpio = health_limpio.rename(columns={col_health: "percentage_health"})
health_limpio = health_limpio[health_limpio["year"].between(2000, 2022)].copy()

print(health_limpio.shape)
display(health_limpio.head())

## 8. Preparar PIB per cápita (`gdp`)

PIB per cápita ajustado por inflación y por diferencias en el costo de vida entre países. Se conserva con el nombre `gdp`.

In [ ]:
columnas_gdp = [c for c in gdp.columns
                if c not in ["entity", "code", "year"]]

print("Columnas candidatas:", columnas_gdp)
assert len(columnas_gdp) == 1, "Revisar manualmente el indicador de PIB per cápita mostrado arriba"

col_gdp = columnas_gdp[0]

gdp_limpio = gdp[["entity", "year", col_gdp]].copy()
gdp_limpio = gdp_limpio.rename(columns={col_gdp: "gdp"})
gdp_limpio = gdp_limpio[gdp_limpio["year"].between(2000, 2022)].copy()

print(gdp_limpio.shape)
display(gdp_limpio.head())

## 9. Revisar llaves y cobertura

Cada combinación `entity`–`year` debe aparecer una sola vez en cada tabla. `code` no se utiliza como llave porque puede faltar en agregados regionales.

In [ ]:
tablas = [("population", population_limpio), ("births", births_limpio),
          ("deaths", deaths_limpio), ("child_deaths", child_deaths_limpio),
          ("health", health_limpio), ("gdp", gdp_limpio)]

for nombre, tabla in tablas:
    duplicados = tabla.duplicated(["entity", "year"]).sum()
    print(nombre, "| entidades:", tabla["entity"].nunique(),
          "| años:", tabla["year"].min(), "a", tabla["year"].max(),
          "| llaves duplicadas:", duplicados)
    assert duplicados == 0, f"Llaves duplicadas en {nombre}"

entidades_comunes = set.intersection(*[set(t["entity"]) for _, t in tablas])

print("Entidades presentes en las seis tablas:", len(entidades_comunes))
print(sorted(entidades_comunes)[:30])

## 10. Unir las tablas, un paso a la vez

Usamos `inner`: permanecen únicamente las combinaciones de entidad y año presentes en ambos DataFrames. `validate="one_to_one"` detecta llaves duplicadas.

In [ ]:
population_births = pd.merge(
    population_limpio, births_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de población + nacimientos:", population_births.shape)
display(population_births.head())

In [ ]:
population_births_deaths = pd.merge(
    population_births, deaths_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de agregar defunciones:", population_births_deaths.shape)
display(population_births_deaths.head())

In [ ]:
demografia = pd.merge(
    population_births_deaths, child_deaths_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Dataset unido:", demografia.shape)
display(demografia.head())

In [ ]:
demografia_health = pd.merge(
    demografia, health_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de agregar gasto en salud:", demografia_health.shape)
print("Entidades:", demografia_health["entity"].nunique())
display(demografia_health.head())

In [ ]:
demografia_final = pd.merge(
    demografia_health, gdp_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Dataset unido:", demografia_final.shape)
print("Entidades:", demografia_final["entity"].nunique())
display(demografia_final.head())

### Entidades que se pierden en la unión

El `inner` descarta las entidades que no aparecen en salud o PIB (por ejemplo, agregados regionales de la ONU). Conviene revisar que no se pierdan países solo por diferencias de nombre (p. ej. `Turkey` vs `Turkiye`).

In [ ]:
en_demografia = set(demografia["entity"])
perdidas = sorted(en_demografia - set(demografia_final["entity"]))
print("Entidades demográficas descartadas:", len(perdidas))
print(perdidas)

## 11. Revisar faltantes y dejar las 8 columnas finales

Eliminamos solo filas con algún indicador necesario ausente y verificamos que la salida tenga exactamente las 8 columnas pedidas.

In [ ]:
columnas_finales = ["entity", "year", "population", "births", "deaths",
                    "child_deaths_under_5", "percentage_health", "gdp"]

demografia = demografia_final.copy()

print("Valores faltantes antes de limpiar:")
print(demografia.isna().sum())
print("Filas antes de limpiar:", len(demografia))

demografia = demografia.dropna(subset=columnas_finales).copy()
demografia = demografia[(demografia["population"] > 0) & (demografia["births"] > 0)].copy()

print("Filas después de limpiar:", len(demografia))
print("Entidades después de limpiar:", demografia["entity"].nunique())
assert not demografia.empty, "No hay entidades y años coincidentes; revisar fuentes y columnas"

In [ ]:
demografia = (demografia[columnas_finales]
              .sort_values(["entity", "year"])
              .reset_index(drop=True))

assert demografia.columns.tolist() == columnas_finales, "Las columnas finales no coinciden"

display(demografia.head())
display(demografia.describe(include="all"))

## 12. Visualizar una entidad disponible

Puedes cambiar `entidad` por cualquier nombre de `demografia["entity"].unique()`. Las tasas se calculan aquí solo para graficar; no forman parte del dataset final.

In [ ]:
entidad = "Mexico" if "Mexico" in demografia["entity"].values else demografia["entity"].iloc[0]
ejemplo = demografia[demografia["entity"] == entidad].sort_values("year").copy()

ejemplo["birth_rate_per_1000"] = ejemplo["births"] / ejemplo["population"] * 1000
ejemplo["death_rate_per_1000"] = ejemplo["deaths"] / ejemplo["population"] * 1000

ejemplo.plot(x="year", y=["birth_rate_per_1000", "death_rate_per_1000"],
             marker="o", figsize=(9, 4), title=f"Tasas demográficas: {entidad}")
plt.ylabel("Por 1,000 habitantes")
plt.grid(alpha=0.3)
plt.show()

## 13. Guardar el resultado

El CSV se guarda junto al notebook. Contiene solamente entidades y años compartidos por las seis fuentes, con las 8 columnas finales.

In [ ]:
from pathlib import Path

carpeta_salida = Path("../data/processed")
carpeta_salida.mkdir(parents=True, exist_ok=True)

salida = carpeta_salida / "demografia_salud_owid_2000_2022.csv"
demografia.to_csv(salida, index=False)

print(f"Archivo guardado: {salida.resolve()} | filas: {len(demografia)} | columnas: {list(demografia.columns)}")